In [ ]:
%load_ext autoreload
%autoreload 2
import os

import numpy as np
import torch

from compactreasoningmodels.datasets import NonogramDataset
from compactreasoningmodels.datasets.collate import collate_raw

if "original_dir" not in globals():
    original_dir = os.getcwd()

os.chdir(os.path.join(original_dir, ".."))
os.environ["DATA_DIR"] = os.path.join(os.getcwd(), "data")
os.environ["MODEL_DIR"] = os.path.join(os.getcwd(), "models")

In [ ]:
dataset = NonogramDataset("traces/large_dataset.jsonl")
dataloader = torch.utils.data.DataLoader(
    dataset, batch_size=1, shuffle=True, collate_fn=collate_raw
)

In [ ]:
from itertools import chain

import pandas as pd

puzzle_idx_counter = 0
puzzle_lookup = {}
runs_list = []
steps_list = []

for X, y, meta in chain.from_iterable(
    zip(X_batch, y_batch, meta_batch) for X_batch, y_batch, meta_batch in dataloader
):
    puzzle_idx = puzzle_idx_counter
    puzzle_idx_counter += 1
    puzzle_lookup[puzzle_idx] = (X, y)

    traces = meta["traces"]
    shape = meta["shape"]
    density = meta["density"]
    mean_clue_runs = meta["mean_clue_runs"]

    for solver_name, solver in traces.items():
        for sr_name, sr in solver.items():
            sampling_ratio = float(sr_name)
            for i, trace in enumerate(sr):
                run_idx = len(runs_list)  # global synthetic run id

                runs_list.append(
                    {
                        "run_idx": run_idx,
                        "puzzle_idx": puzzle_idx,
                        "shape": shape,
                        "density": density,
                        "mean_clue_runs": mean_clue_runs,
                        "solver": solver_name,
                        "sampling_ratio": sampling_ratio,
                        "run_index": i,
                        "solved": trace["solved"],
                        "num_steps": trace["num_steps"],
                        "step_ratio": trace["step_ratio"],
                        "mse_loss_final": trace["mse_losses"][-1] if trace["mse_losses"] else None,
                        "cr_loss_final": trace["cr_losses"][-1] if trace["cr_losses"] else None,
                        "steps_to_solve": trace["steps_to_solve"],
                    }
                )

                for step, cr, mse in zip(trace["steps"], trace["cr_losses"], trace["mse_losses"]):
                    steps_list.append(
                        {
                            "run_idx": run_idx,
                            "step": step,
                            "mse_loss": mse,
                            "cr_loss": cr,
                        }
                    )

df_runs = pd.DataFrame(runs_list)
df_steps = pd.DataFrame(steps_list)

In [ ]:
summary = df_runs.groupby(["solver", "sampling_ratio"]).agg(
    max_steps_to_solve=("steps_to_solve", "max"),
    mean_steps_to_solve=("steps_to_solve", lambda x: x[x != -1].mean()),
)
print(summary)

In [ ]:
summary = (
    df_runs.groupby(["solver", "sampling_ratio"])
    .agg(
        accuracy_mean=("solved", "mean"),
        accuracy_std=("solved", "std"),
        mse_mean=("mse_loss_final", "mean"),
        mse_std=("mse_loss_final", "std"),
        cr_mean=("cr_loss_final", "mean"),
        cr_std=("cr_loss_final", "std"),
        n_runs=("run_idx", "count"),
    )
    .reset_index()
)
print(summary)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# --- 1. Pivot: one row per (solver, sampling_ratio) config, one column per puzzle ---
pivot = df_runs.pivot_table(
    index=["solver", "sampling_ratio"],
    columns="puzzle_idx",
    values="solved",
    aggfunc="mean",
)

print(f"Shape: {pivot.shape[0]} configs x {pivot.shape[1]} puzzles")
print(f"Missing values: {pivot.isna().sum().sum()} of {pivot.size}")

# --- 2. Handle missing puzzle coverage + standardize ---
# Standardizing matters here: puzzles vary wildly in difficulty, so without
# scaling, PC1 would mostly reflect "which puzzles are hardest" rather than
# "how configs differ from each other" — same effect but on the puzzle axis now.
imputer = SimpleImputer(strategy="mean")
X = imputer.fit_transform(pivot.values)
X = StandardScaler().fit_transform(X)

# --- 3. PCA to 2D ---
pca = PCA(n_components=2)
coords = pca.fit_transform(X)

print(
    f"Explained variance: PC1={pca.explained_variance_ratio_[0]:.1%}, "
    f"PC2={pca.explained_variance_ratio_[1]:.1%}"
)

result = pivot.index.to_frame(index=False)
result["pc1"] = coords[:, 0]
result["pc2"] = coords[:, 1]

# --- 4. Plot: each point = one (solver, sampling_ratio) config ---
fig, ax = plt.subplots(figsize=(8, 6))
solvers = result["solver"].unique()
cmap = plt.get_cmap("tab10")

for i, solver in enumerate(solvers):
    subset = result[result.solver == solver]
    ax.scatter(subset["pc1"], subset["pc2"], color=cmap(i % 10), s=80, label=solver)
    for _, row in subset.iterrows():
        ax.annotate(
            f"sr={row['sampling_ratio']}",
            (row["pc1"], row["pc2"]),
            fontsize=8,
            xytext=(4, 4),
            textcoords="offset points",
        )

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} var)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} var)")
ax.set_title("PCA of (solver, sampling_ratio) configs across puzzles, by MSE")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
from collections import defaultdict
from itertools import chain

import pandas as pd

N_BUCKETS = 10
bucket_edges = np.linspace(0, 1, N_BUCKETS + 1)

# accumulator: (solver, sampling_ratio) -> dict of step_idx -> length-10 count array
dist_accum = defaultdict(lambda: defaultdict(lambda: np.zeros(N_BUCKETS, dtype=np.int64)))

for X, y, meta in chain.from_iterable(
    zip(X_batch, y_batch, meta_batch) for X_batch, y_batch, meta_batch in dataloader
):
    traces = meta["traces"]

    for solver_name, solver in traces.items():
        for sr_name, sr in solver.items():
            sampling_ratio = float(sr_name)
            key = (solver_name, sampling_ratio)

            for trace in sr:  # each trace = one run
                for step_idx, grid in enumerate(trace["steps"]):
                    grid_arr = np.asarray(grid)
                    counts, _ = np.histogram(grid_arr.ravel(), bins=bucket_edges)
                    dist_accum[key][step_idx] += counts

# --- Stack into a 2D array per (solver, sampling_ratio): shape (num_steps, 10) ---
dist_matrices = {}
for key, step_dict in dist_accum.items():
    max_step = max(step_dict.keys())
    matrix = np.zeros((max_step + 1, N_BUCKETS), dtype=np.int64)
    for step_idx, counts in step_dict.items():
        matrix[step_idx] = counts
    dist_matrices[key] = matrix

# Example: inspect one config
solver, sampling_ratio = list(dist_matrices.keys())[0]
print(f"{solver}, sr={sampling_ratio}: shape {dist_matrices[(solver, sampling_ratio)].shape}")
print(dist_matrices[(solver, sampling_ratio)])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def plot_distribution_heatmap(matrix, title, normalize=True, vmin=0, vmax=1):
    if normalize:
        # normalize each step's row to sum to 1 (since later steps have fewer contributing runs)
        row_sums = matrix.sum(axis=1, keepdims=True)
        matrix = np.divide(
            matrix, row_sums, out=np.zeros_like(matrix, dtype=float), where=row_sums != 0
        )

    fig, ax = plt.subplots(figsize=(8, 5))
    im = ax.imshow(
        matrix.T,
        aspect="auto",
        origin="lower",
        cmap="viridis",
        extent=[0, matrix.shape[0], 0, 1],
        vmin=vmin,
        vmax=vmax,
    )
    ax.set_xlabel("Step")
    ax.set_ylabel("Cell value bucket (0-1)")
    ax.set_title(title)
    plt.colorbar(im, label="proportion of cells" if normalize else "cell count")
    plt.tight_layout()
    plt.show()


for (solver, sampling_ratio), matrix in dist_matrices.items():
    plot_distribution_heatmap(matrix, f"{solver}, sampling_ratio={sampling_ratio}")

In [ ]:
from collections import defaultdict

import numpy as np
import pandas as pd

results = defaultdict(lambda: {"accuracy": [], "mse_loss": [], "cr_loss": []})

for clues, grid, meta in dataloader:
    true_grid = grid[0]
    traces = meta[0].get("traces", None)
    if not traces:
        continue

    for trace_name, sampling_ratios in traces.items():
        for sampling_ratio, runs in sampling_ratios.items():
            key = (trace_name, sampling_ratio)
            results[key]["accuracy"].append(np.mean([run["solved"] for run in runs]))
            results[key]["mse_loss"].append(np.mean([run["mse_losses"][-1] for run in runs]))
            results[key]["cr_loss"].append(np.mean([run["cr_losses"][-1] for run in runs]))

# Build a tidy summary table: mean ± std for each metric
rows = []
for (trace_name, sampling_ratio), metrics in results.items():
    row = {"trace": trace_name, "sampling_ratio": sampling_ratio}
    for metric_name, values in metrics.items():
        row[f"{metric_name}_mean"] = np.mean(values)
        row[f"{metric_name}_std"] = np.std(values)
    rows.append(row)

df = pd.DataFrame(rows).sort_values(["trace", "sampling_ratio"]).reset_index(drop=True)
df

In [ ]:
from collections import defaultdict
from itertools import product

import numpy as np
import pandas as pd

# metric_name -> (run_key, how to extract the value from run[run_key])
metric_extractors = {
    "accuracy": ("solved", lambda v: v),  # scalar per run
    "mse_loss": ("mse_losses", lambda v: v[-1]),  # last-step loss
    "cr_loss": ("cr_losses", lambda v: v[-1]),  # last-step loss
}
metrics = list(metric_extractors.keys())

data = defaultdict(lambda: defaultdict(list))

for clues, grid, meta in dataloader:
    traces = meta[0].get("traces", None)
    if not traces:
        continue
    for trace_name, sampling_ratios in traces.items():
        for sampling_ratio, runs in sampling_ratios.items():
            if not runs:
                continue
            key = (trace_name, sampling_ratio)
            for metric_name, (run_key, extract) in metric_extractors.items():
                run_values = [extract(run[run_key]) for run in runs]
                data[key][metric_name].append(run_values)

keys = sorted(data.keys())
key_labels = {k: f"{k[0]}|{k[1]}" for k in keys}


def paired_values(key_a, key_b, metric):
    a_vals, b_vals = [], []
    for runs_a, runs_b in zip(data[key_a][metric], data[key_b][metric]):
        for i, va in enumerate(runs_a):
            for j, vb in enumerate(runs_b):
                if key_a == key_b and i == j:
                    continue
                a_vals.append(va)
                b_vals.append(vb)
    return np.array(a_vals), np.array(b_vals)


labels = [key_labels[k] for k in keys]
corr_matrices = {
    metric: pd.DataFrame(index=labels, columns=labels, dtype=float) for metric in metrics
}

for metric in metrics:
    for k1, k2 in product(keys, keys):
        a, b = paired_values(k1, k2, metric)
        if len(a) < 2 or np.std(a) == 0 or np.std(b) == 0:
            val = np.nan
        else:
            val = np.corrcoef(a, b)[0, 1]
        corr_matrices[metric].loc[key_labels[k1], key_labels[k2]] = val

for metric in metrics:
    print(f"Correlation matrix ({metric}):")
    with pd.option_context("display.max_rows", None, "display.max_columns", None):
        display(corr_matrices[metric])

In [ ]:
# create new dataset with puzzle index and mse, filter to only do sampling ratio = 1.0
df_filtered = df_runs.copy()
df_filtered = df_runs[df_runs["sampling_ratio"] == 1.0]
df_filtered = df_filtered[df_filtered["mse_loss_final"] != 0]

# Step 1: Get top 10 puzzle_idx per solver as sets
puzzle_sets = (
    df_filtered.sort_values("mse_loss_final", ascending=False)
    .groupby("solver", sort=False)
    .head(10)
    .groupby("solver")["puzzle_idx"]
    .apply(set)
    .to_dict()
)
solver_names = sorted(puzzle_sets.keys())
jaccard_matrix = pd.DataFrame(index=solver_names, columns=solver_names)

for s1 in solver_names:
    for s2 in solver_names:
        if s1 == s2:
            jaccard_matrix.loc[s1, s2] = 1.0
        else:
            union = puzzle_sets[s1] | puzzle_sets[s2]
            jaccard_matrix.loc[s1, s2] = len(puzzle_sets[s1] & puzzle_sets[s2]) / len(union)
jaccard_matrix

## Comparing Solving Traces

In [ ]:
mse = (
    df_runs[["solver", "mse_loss_final"]]
    .dropna(subset=["mse_loss_final"])
    .groupby("solver")["mse_loss_final"]
)
print(mse.get_group("mac").iloc[:50].tolist())

In [ ]:
from itertools import combinations

import numpy as np


def weighted_kendall_tau(order_a, order_b, values):
    """
    order_a, order_b: arrays of record IDs in sorted order
    values: dict or array mapping record ID -> value
    """
    # Rank of each item in each list
    rank_a = {item: r for r, item in enumerate(order_a)}
    rank_b = {item: r for r, item in enumerate(order_b)}

    concordant_weight = 0
    discordant_weight = 0

    for i, j in combinations(order_a, 2):
        w = abs(values[i] - values[j])  # or another weight function

        # Same relative order in both lists?
        if (rank_a[i] - rank_a[j]) * (rank_b[i] - rank_b[j]) > 0:
            concordant_weight += w
        else:
            discordant_weight += w

    return (concordant_weight - discordant_weight) / (concordant_weight + discordant_weight)


print("Weighted Kendall's tau between 'mac' and 'gradient_descent_global_sgd':")
scores = (
    df_runs[df_runs["sampling_ratio"] == 1.0]
    .groupby(["solver", "puzzle_idx"])["mse_loss_final"]
    .mean()
)

mac_scores = scores.loc["model_solver"].dropna()
gd_scores = scores.loc["backtracking_search"].dropna()
# normalise
mac_scores = (mac_scores - mac_scores.min()) / (mac_scores.max() - mac_scores.min())
gd_scores = (gd_scores - gd_scores.min()) / (gd_scores.max() - gd_scores.min())

common = mac_scores.index.intersection(gd_scores.index)

order_a = mac_scores.loc[common].sort_values(ascending=False).index.tolist()
order_b = gd_scores.loc[common].sort_values(ascending=False).index.tolist()
values = mac_scores.loc[common].to_dict()

print(weighted_kendall_tau(order_a, order_b, values))

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import rankdata


def weighted_kendall_tau(order_a, order_b, values_a, values_b):
    """
    Weighted Kendall's tau: penalizes swaps proportional to value difference.
    Returns similarity in [-1, 1], where 1 = identical ordering.
    """
    rank_a = {rid: r for r, rid in enumerate(order_a)}
    rank_b = {rid: r for r, rid in enumerate(order_b)}

    concordant_w = discordant_w = 0.0

    for i, j in combinations(order_a, 2):
        # Weight = how different are these records in value space
        w = (abs(values_a[i] - values_a[j]) + abs(values_b[i] - values_b[j])) / 2

        if (rank_a[i] - rank_a[j]) * (rank_b[i] - rank_b[j]) > 0:
            concordant_w += w
        else:
            discordant_w += w

    total = concordant_w + discordant_w
    return (concordant_w - discordant_w) / total if total > 0 else 1.0


def solver_similarity_matrix(
    df, value_col, solver_col="solver", instance_col="instance", normalize="rank"
):
    """
    Pairwise weighted rank correlation between solvers.

    normalize: "rank" (robust), "minmax", "zscore", or None
    """
    solver_data = {}

    for solver in df[solver_col].unique():
        subset = df.loc[df[solver_col] == solver, [instance_col, value_col]].dropna()
        if len(subset) < 2:
            continue

        record_ids = subset[instance_col].values
        vals = subset[value_col].values

        # Normalize (crucial for different scales!)
        if normalize == "rank":
            norm_vals = rankdata(vals) / len(vals)
        elif normalize == "minmax":
            rng = vals.max() - vals.min()
            norm_vals = (vals - vals.min()) / rng if rng > 0 else np.zeros_like(vals)
        elif normalize == "zscore":
            std = vals.std()
            norm_vals = (vals - vals.mean()) / std if std > 0 else np.zeros_like(vals)
        else:
            norm_vals = vals

        sorted_idx = np.argsort(norm_vals)
        solver_data[solver] = {
            "order": [record_ids[i] for i in sorted_idx],
            "values": dict(zip(record_ids, norm_vals)),
        }

    # Pairwise comparison
    solvers = sorted(solver_data.keys())
    sim_matrix = pd.DataFrame(np.eye(len(solvers)), index=solvers, columns=solvers)

    for s1, s2 in combinations(solvers, 2):
        common = sorted(set(solver_data[s1]["order"]) & set(solver_data[s2]["order"]))
        if len(common) < 2:
            sim_matrix.loc[s1, s2] = sim_matrix.loc[s2, s1] = np.nan
            continue

        order_a = [r for r in solver_data[s1]["order"] if r in common]
        order_b = [r for r in solver_data[s2]["order"] if r in common]

        sim = weighted_kendall_tau(
            order_a, order_b, solver_data[s1]["values"], solver_data[s2]["values"]
        )
        sim_matrix.loc[s1, s2] = sim_matrix.loc[s2, s1] = sim

    return sim_matrix


# Usage
df_similarity = (
    df_runs.loc[df_runs["sampling_ratio"] == 1.0]
    .groupby(["solver", "puzzle_idx"], as_index=False)["solved"]
    .mean()
)

sim_matrix = solver_similarity_matrix(
    df_similarity,
    value_col="solved",
    solver_col="solver",
    instance_col="puzzle_idx",
    normalize="rank",
)

print(sim_matrix.round(3))

In [ ]:
import numpy as np

steps_list = {name: [] for name in solver_names}
for X, y, meta in chain.from_iterable(
    zip(X_batch, y_batch, meta_batch) for X_batch, y_batch, meta_batch in dataloader
):
    traces = meta["traces"]
    shape = meta["shape"]
    density = meta["density"]
    mean_clue_runs = meta["mean_clue_runs"]

    for solver_name, solver in traces.items():
        if solver_name not in solver_names:
            continue
        sr = solver.get("1.0", [])
        maxtrix = []
        steps_list[solver_name].append(np.array(sr[0]["steps"]))
for i in range(len(steps_list["mac"])):
    steps_list["mac"][i] = steps_list["mac"][i][:20]

In [ ]:
import numpy as np
from scipy import signal


def _morlet2(M, s, w=5.0):
    """
    Standalone replacement for the removed scipy.signal.morlet2.
    Complex Morlet wavelet, same definition scipy used:
        output = pi**-0.25 * exp(1j*w*x/s) * exp(-0.5*(x/s)**2)
    normalized and scaled by width s.
    """
    x = np.arange(0, M) - (M - 1.0) / 2
    x = x / s
    wavelet = np.pi**-0.25 * np.exp(1j * w * x) * np.exp(-0.5 * x**2)
    output = np.sqrt(1 / s) * wavelet
    return output


def cwt_morlet(sig, widths, w0=5.0):
    """
    Manual replacement for the removed scipy.signal.cwt, using a
    locally-defined Morlet wavelet (scipy.signal.morlet2 is also gone).
    """
    output = np.empty((len(widths), len(sig)), dtype=np.complex128)
    for ind, width in enumerate(widths):
        N = np.min([10 * width, len(sig)])
        wavelet_data = _morlet2(N, width, w=w0)
        output[ind] = signal.convolve(sig, wavelet_data, mode="same")
    return output


def simple_scattering_like_features(trace, widths=None):
    if widths is None:
        steps = trace.shape[0]
        widths = np.unique(np.logspace(0, np.log10(max(4, steps // 4)), 20).astype(int))
        widths = np.clip(widths, 1, steps // 2)

    features = []
    _, width = trace.shape

    for w in range(width):
        sig = trace[:, w]
        coeffs = cwt_morlet(sig, widths)

        # Use magnitude for all statistics (Morlet is complex)
        mag = np.abs(coeffs)

        # Global temporal pooling
        features.extend(mag.mean(axis=1))
        features.extend(mag.std(axis=1))
        features.extend(np.percentile(mag, [25, 50, 75], axis=1).flatten())

        # Second-order: correlations between scales (on magnitude, safe from zero-std)
        # Replace zero-std rows with zeros BEFORE corrcoef
        stds = mag.std(axis=1)
        safe_mag = mag.copy()
        zero_std_mask = stds < 1e-12
        safe_mag[zero_std_mask, :] = 0.0

        if np.all(zero_std_mask):
            # All scales are constant; add zeros for correlation features
            n_scales = len(widths)
            triu_size = n_scales * (n_scales - 1) // 2
            features.extend(np.zeros(triu_size))
        else:
            scale_corr = np.corrcoef(safe_mag)
            # Sanitize any remaining NaN/Inf from numerical edge cases
            scale_corr = np.nan_to_num(scale_corr, nan=0.0, posinf=0.0, neginf=0.0)
            triu_idx = np.triu_indices_from(scale_corr, k=1)
            features.extend(scale_corr[triu_idx])

    return np.array(features, dtype=np.float64)


def linear_cka(X, Y):
    """Linear CKA — now fully real-valued."""
    X = np.real(X) - np.real(X).mean(axis=0, keepdims=True)
    Y = np.real(Y) - np.real(Y).mean(axis=0, keepdims=True)
    Kx = X @ X.T
    Ky = Y @ Y.T
    Kxy = X @ Y.T
    numerator = (Kxy**2).sum()
    denominator = np.sqrt((Kx**2).sum() * (Ky**2).sum())
    return numerator / denominator if denominator > 1e-12 else 0.0


def embed_all_solvers(solvers_dict, widths=None):
    """
    Embed all traces from all solvers.

    solvers_dict: {solver_name: [trace1, trace2, ...]}
    Returns: {solver_name: np.array of shape (n_samples, embedding_dim)}
    """
    embeddings = {}

    for solver_name, traces in solvers_dict.items():
        solver_embs = []
        for trace in traces:
            emb = simple_scattering_like_features(trace, widths=widths)
            solver_embs.append(emb)
        embeddings[solver_name] = np.stack(solver_embs)
        print(f"{solver_name}: {len(traces)} samples → {embeddings[solver_name].shape}")

    return embeddings


# =============================================================================
# COMPARISON
# =============================================================================


def pairwise_similarity(embeddings):
    solver_names = list(embeddings.keys())
    n = len(solver_names)
    sim_matrix = np.zeros((n, n))

    for i in range(n):
        for j in range(n):
            sim_matrix[i, j] = linear_cka(embeddings[solver_names[i]], embeddings[solver_names[j]])

    # Print
    print(f"{'':15}", end="")
    for name in solver_names:
        print(f"{name:15}", end="")
    print()
    for i, name_i in enumerate(solver_names):
        print(f"{name_i:15}", end="")
        for j in range(n):
            print(f"{sim_matrix[i, j]:15.3f}", end="")
        print()

    return sim_matrix, solver_names


# =============================================================================
# USAGE
# =============================================================================

# Your data
# steps_list = {
#     'solver_A': [trace1, trace2, ...],  # each (steps, width, height)
#     'solver_B': [...],
# }


# If traces are 3D, flatten spatial first:
def flatten_spatial(trace):
    if trace.ndim == 3:
        steps, w, h = trace.shape
        return trace.reshape(steps, w * h)
    return trace


# Flatten if needed
steps_list_flat = {
    name: [flatten_spatial(t) for t in traces] for name, traces in steps_list.items()
}

# Compute embeddings
embeddings = embed_all_solvers(steps_list_flat)

# Compare
sim_matrix, names = pairwise_similarity(embeddings)

In [ ]:
# Add debugging before pairwise_similarity:
for name, emb in embeddings.items():
    print(
        f"{name}: shape={emb.shape}, has_nan={np.isnan(emb).any()}, "
        f"has_inf={np.isinf(emb).any()}, all_zeros={np.allclose(emb, 0)}"
    )

In [ ]:
import numpy as np


def morlet_wavelet_bank(N, J, Q):
    """Generate a bank of Morlet wavelet filters in the frequency domain."""
    filters = []
    for j in range(J):
        for q in range(Q):
            xi = 0.4 * 2 ** (-(j + q / Q))  # center frequency
            sigma = 0.1 * 2 ** (j + q / Q)  # bandwidth
            omega = np.fft.fftfreq(N)
            psi_hat = np.exp(-0.5 * ((omega - xi) / sigma) ** 2)
            filters.append(psi_hat)
    return filters


def lowpass_filter(N, J):
    """Gaussian lowpass (scaling function) for averaging."""
    omega = np.fft.fftfreq(N)
    sigma = 0.1 * 2**J
    return np.exp(-0.5 * (omega / sigma) ** 2)


def scattering_order1(x, J, Q):
    N = len(x)
    x_hat = np.fft.fft(x)

    filters = morlet_wavelet_bank(N, J, Q)
    phi = lowpass_filter(N, J)

    S0 = np.abs(np.fft.ifft(x_hat * phi))  # 0th order: smoothed signal
    S1 = []
    for psi_hat in filters:
        # Wavelet convolution
        conv = np.fft.ifft(x_hat * psi_hat)
        # Modulus (non-linearity)
        modulus = np.abs(conv)
        # Low-pass + subsample (averaging = invariance)
        smoothed = np.fft.ifft(np.fft.fft(modulus) * phi)
        S1.append(np.abs(smoothed))

    return S0, np.array(S1)


# Usage
x = np.sin(2 * np.pi * 0.05 * np.arange(1024)) + 0.1 * np.random.randn(1024)
S0, S1 = scattering_order1(x, J=4, Q=4)

In [ ]:
print(S0[0])  # (1024,)